# Airline Satisfaction — CatBoost baseline

This notebook trains a binary classifier for `satisfaction`, evaluates it with **ROC AUC** on a stratified holdout, then fits on all labeled rows and writes `submission_catboost.csv`. It assumes this notebook and the two CSV files are in the same folder.


In [2]:
# Run once if CatBoost is missing from this notebook environment.
%pip install catboost


Note: you may need to restart the kernel to use updated packages.


In [13]:
from pathlib import Path
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix


In [14]:
# Files are next to this notebook. The training CSV was named "train (1).csv".
DATA_DIR = Path.cwd()
TRAIN_PATH = DATA_DIR / "train (1).csv"
TEST_PATH = DATA_DIR / "test.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
TARGET = "satisfaction"
ID_COL = "id"

print("train:", train.shape, "test:", test.shape)
display(train.head())
print("\nTarget distribution:")
print(train[TARGET].value_counts(dropna=False))
print("\nMissing values (training):")
print(train.isna().sum().sort_values(ascending=False).head(10))


train: (699635, 23) test: (299844, 22)


,id,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,...,Baggage handling,Checkin service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,Gender,Customer Type,Type of Travel,Class,satisfaction
0,0,36,694,4,4,4,3,1,4,1,...,4,3,1,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
1,1,56,651,5,5,5,5,5,4,4,...,5,4,3,0,0.0,Male,Loyal Customer,Business travel,Business,True
2,2,31,1371,3,4,2,5,3,3,3,...,4,5,3,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
3,3,10,1616,3,5,3,3,3,3,3,...,3,4,3,0,0.0,Male,Loyal Customer,Personal Travel,Eco,False
4,4,23,481,3,4,3,4,5,3,5,...,5,3,5,5,0.0,Female,disloyal Customer,Business travel,Eco,False



Target distribution:
satisfaction
False    389296
True     310339
Name: count, dtype: int64

Missing values (training):
Arrival Delay in Minutes      292
id                              0
Leg room service                0
Class                           0
Type of Travel                  0
Customer Type                   0
Gender                          0
Departure Delay in Minutes      0
Cleanliness                     0
Checkin service                 0
dtype: int64


In [15]:
# Separate identifiers and target; do not train on the ID by default.
X = train.drop(columns=[TARGET, ID_COL])
y = train[TARGET].astype(int)
X_test = test.drop(columns=[ID_COL]).copy()

# CatBoost accepts categorical columns directly; represent missing categories explicitly.
cat_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
for col in cat_features:
    X[col] = X[col].fillna("__MISSING__").astype(str)
    X_test[col] = X_test[col].fillna("__MISSING__").astype(str)

# Ensure numeric columns use consistent numeric values and preserve missing values for CatBoost.
for col in X.columns.difference(cat_features):
    X[col] = pd.to_numeric(X[col], errors="coerce")
    X_test[col] = pd.to_numeric(X_test[col], errors="coerce")

print("Categorical features:", cat_features)
print("Positive class rate:", y.mean())


Categorical features: ['Gender', 'Customer Type', 'Type of Travel', 'Class']
Positive class rate: 0.44357272006117476


In [16]:
# Train the current best feature setup: original features plus Travel_Context.
# This uses one fixed stratified holdout so its AUC remains comparable to the earlier runs.
from sklearn.model_selection import train_test_split

row_indices = np.arange(len(y))
train_idx, valid_idx = train_test_split(
    row_indices, test_size=0.2, random_state=42, stratify=y
)
y_train = y.iloc[train_idx]
y_valid = y.iloc[valid_idx]

context_cols = ["Type of Travel", "Customer Type", "Class"]
X_final = X.copy()
X_test_final = X_test.copy()
X_final["Travel_Context"] = X_final[context_cols].astype(str).agg(" | ".join, axis=1)
X_test_final["Travel_Context"] = X_test_final[context_cols].astype(str).agg(" | ".join, axis=1)
final_cat_features = cat_features + ["Travel_Context"]
best_depth = 7

model = CatBoostClassifier(
    iterations=4000,
    learning_rate=0.03,
    depth=best_depth,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=300,
    early_stopping_rounds=150,
    allow_writing_files=False,
    thread_count=-1,
)
model.fit(
    X_final.iloc[train_idx], y_train,
    cat_features=final_cat_features,
    eval_set=(X_final.iloc[valid_idx], y_valid),
    use_best_model=True,
)
valid_proba = model.predict_proba(X_final.iloc[valid_idx])[:, 1]
validation_auc = roc_auc_score(y_valid, valid_proba)
print(f"Travel_Context validation ROC AUC: {validation_auc:.6f}")
print(f"Best iteration: {model.best_iteration_}")


0:	test: 0.9418300	best: 0.9418300 (0)	total: 290ms	remaining: 19m 19s
300:	test: 0.9560421	best: 0.9560421 (300)	total: 1m 43s	remaining: 21m 6s
600:	test: 0.9570618	best: 0.9570618 (600)	total: 3m 25s	remaining: 19m 25s
900:	test: 0.9573925	best: 0.9573925 (900)	total: 5m 11s	remaining: 17m 51s
1200:	test: 0.9575417	best: 0.9575417 (1200)	total: 6m 56s	remaining: 16m 11s
1500:	test: 0.9576491	best: 0.9576491 (1500)	total: 8m 43s	remaining: 14m 30s
1800:	test: 0.9577132	best: 0.9577132 (1796)	total: 12m 29s	remaining: 15m 15s
2100:	test: 0.9577827	best: 0.9577843 (2091)	total: 17m 37s	remaining: 15m 55s
2400:	test: 0.9578571	best: 0.9578571 (2400)	total: 19m 34s	remaining: 13m 2s
2700:	test: 0.9578992	best: 0.9578995 (2698)	total: 21m 29s	remaining: 10m 20s
3000:	test: 0.9579327	best: 0.9579377 (2984)	total: 23m 27s	remaining: 7m 48s
3300:	test: 0.9579546	best: 0.9579590 (3273)	total: 25m 20s	remaining: 5m 22s
3600:	test: 0.9579874	best: 0.9579874 (3600)	total: 27m 9s	remaining: 3m
39

## Optional threshold-based check

 The competition is scored by ROC AUC, so probability ranking is the key result. The next cell is only an optional classification report at threshold 0.5.


In [17]:
valid_pred = (valid_proba >= 0.5).astype(int)
print(classification_report(y_valid, valid_pred, digits=4))
print("Confusion matrix:\n", confusion_matrix(y_valid, valid_pred))


              precision    recall  f1-score   support

           0     0.9239    0.9436    0.9336     77859
           1     0.9273    0.9025    0.9147     62068

    accuracy                         0.9254    139927
   macro avg     0.9256    0.9231    0.9242    139927
weighted avg     0.9254    0.9254    0.9253    139927

Confusion matrix:
 [[73467  4392]
 [ 6050 56018]]


## 5-fold stratified cross-validation

 Cross-validation estimates how consistently the current CatBoost setup performs across different held-out portions of labeled data. In 5-fold CV, the data is split into five parts. The model trains five times: each run trains on four parts (80%) and validates on the remaining part (20%), rotating the validation part each time. We report each fold ROC AUC, their mean and standard deviation, and a combined out-of-fold AUC. `test.csv` is not used.

 This evaluates the selected feature set (original features plus `Travel_Context`) with the current CatBoost settings. It can take several times longer than one fit because it trains five models.


In [19]:
from sklearn.model_selection import StratifiedKFold

# Use a repeatable, label-balanced five-fold split.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_fold_aucs = []
cv_best_iterations = []

# Each row gets validation predictions from the fold where it was held out.
oof_proba = np.zeros(len(y), dtype=float)

for fold_number, (fit_idx, holdout_idx) in enumerate(cv.split(X_final, y), start=1):
    print(f"\nFold {fold_number}/5")
    fold_model = CatBoostClassifier(
        iterations=4000,
        learning_rate=0.03,
        depth=best_depth,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42 + fold_number,
        verbose=300,
        early_stopping_rounds=150,
        allow_writing_files=False,
        thread_count=-1,
    )
    fold_model.fit(
        X_final.iloc[fit_idx], y.iloc[fit_idx],
        cat_features=final_cat_features,
        eval_set=(X_final.iloc[holdout_idx], y.iloc[holdout_idx]),
        use_best_model=True,
    )

    fold_proba = fold_model.predict_proba(X_final.iloc[holdout_idx])[:, 1]
    oof_proba[holdout_idx] = fold_proba
    fold_auc = roc_auc_score(y.iloc[holdout_idx], fold_proba)
    cv_fold_aucs.append(fold_auc)
    cv_best_iterations.append(fold_model.best_iteration_ + 1)
    print(f"Fold {fold_number} ROC AUC: {fold_auc:.6f}; best trees: {cv_best_iterations[-1]}")

cv_mean_auc = float(np.mean(cv_fold_aucs))
cv_std_auc = float(np.std(cv_fold_aucs, ddof=1))
oof_auc = roc_auc_score(y, oof_proba)
print("\nFold AUCs:", [round(score, 6) for score in cv_fold_aucs])
print(f"Mean 5-fold ROC AUC: {cv_mean_auc:.6f} +/- {cv_std_auc:.6f}")
print(f"Combined out-of-fold ROC AUC: {oof_auc:.6f}")

# Save fold-by-fold scores and training lengths so the CV result is recorded in a file.
cv_results = pd.DataFrame({
    "fold": np.arange(1, 6),
    "roc_auc": cv_fold_aucs,
    "best_trees": cv_best_iterations,
})
cv_results.loc[len(cv_results)] = ["mean", cv_mean_auc, np.mean(cv_best_iterations)]
cv_results_path = DATA_DIR / "catboost_cv_results.csv"
cv_results.to_csv(cv_results_path, index=False)
print("Saved CV results:", cv_results_path.resolve())



Fold 1/5
0:	test: 0.9426474	best: 0.9426474 (0)	total: 284ms	remaining: 18m 56s
300:	test: 0.9570674	best: 0.9570674 (300)	total: 1m 47s	remaining: 21m 59s
600:	test: 0.9580567	best: 0.9580567 (600)	total: 3m 48s	remaining: 21m 31s
900:	test: 0.9583988	best: 0.9583988 (900)	total: 5m 49s	remaining: 20m 2s
1200:	test: 0.9586000	best: 0.9586000 (1200)	total: 7m 52s	remaining: 18m 20s
1500:	test: 0.9587184	best: 0.9587184 (1500)	total: 9m 49s	remaining: 16m 20s
1800:	test: 0.9588288	best: 0.9588310 (1794)	total: 12m 43s	remaining: 15m 31s
2100:	test: 0.9588657	best: 0.9588678 (2096)	total: 16m 48s	remaining: 15m 11s
2400:	test: 0.9589141	best: 0.9589157 (2391)	total: 19m 32s	remaining: 13m
2700:	test: 0.9589595	best: 0.9589603 (2694)	total: 21m 24s	remaining: 10m 17s
3000:	test: 0.9590007	best: 0.9590007 (3000)	total: 23m 31s	remaining: 7m 49s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 0.9590048342
bestIteration = 3085

Shrink model to first 3086 iterations.
Fold 

In [ ]:
# Refit on all labeled rows using the feature set selected by validation.
best_iterations = model.best_iteration_ + 1 if model.best_iteration_ is not None else None
if best_iterations is None or best_iterations < 1:
    best_iterations = 1500

final_model = CatBoostClassifier(
    iterations=best_iterations,
    learning_rate=0.03,
    depth=best_depth,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=200,
    allow_writing_files=False,
    thread_count=-1,
)
final_model.fit(X_final, y, cat_features=final_cat_features)
test_proba = final_model.predict_proba(X_test_final)[:, 1]

submission = pd.DataFrame({ID_COL: test[ID_COL], TARGET: test_proba})
submission_path = DATA_DIR / "submission_catboost.csv"
submission.to_csv(submission_path, index=False)
print("Saved:", submission_path.resolve())
print(submission.head())
print("Rows:", len(submission), "IDs unique:", submission[ID_COL].is_unique)
print("Probabilities in [0, 1]:", submission[TARGET].between(0, 1).all())


0:	total: 319ms	remaining: 21m 14s
